# 07 — Tessera 128x128 patches

Uses `GeoTesseraZarr.read_patch(lon, lat, year, size_px)` (confirmed via introspection, geotessera 0.10.2) — handles UTM-zone-boundary merging internally; `open_zone` caches so this costs at most 2 zone-opens for West Bengal.

`crs`/`transform` are stored **per sample**, not per tile — a zone-seam patch gets its own merged projection. `SCALE_FACTOR` is measured from a real probe (Tessera's range is not [-1,1] like AlphaEarth's).

Needs `zarr>=3` (geotessera imports v3-only modules) — the S1/S2/AlphaEarth notebooks need `zarr<3`. Run this in its own environment.

Only 2024 reliably covers this AOI (measured below, not assumed from `gz.years`).

In [1]:
!pip -q install -U geotessera geopandas "zarr>=3" rioxarray pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.9/258.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.4 MB/s eta 0:00:00


In [2]:
import os, json, glob, time, shutil, inspect, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import zarr
assert int(zarr.__version__.split('.')[0]) >= 3, (
    f"zarr {zarr.__version__} installed -- this notebook needs zarr 3.x. "
    "geotessera's own code imports zarr.abc.store and zarr.storage.ObjectStore, which "
    "only exist in zarr 3.x, so this requirement comes from geotessera, not a choice here. "
    "IMPORTANT: this is the OPPOSITE requirement from the S1/S2 and AlphaEarth patch "
    "notebooks, which need zarr 2.x. Do not run this notebook in the same runtime as "
    "those without restarting first -- Runtime -> Restart session, run the pip cell, "
    "then this cell again.")
print("zarr version OK:", zarr.__version__)

# Self-test: write + read a tiny dummy array with the exact API process_tile() uses below,
# BEFORE trusting it on real tiles. zarr 3.x's array-creation API differs across its own
# point releases, so this is verified here rather than assumed.
from zarr.codecs import BytesCodec
try:
    from zarr.codecs import BloscCodec, BloscShuffle
    _codecs = [BytesCodec(), BloscCodec(cname='zstd', clevel=5, shuffle=BloscShuffle.bitshuffle)]
except ImportError:
    _codecs = None

_test_path = '/content/_zarr_selftest.zarr'
import shutil as _sh
if os.path.exists(_test_path):
    _sh.rmtree(_test_path)

_dummy = (np.random.rand(3, 2, 4, 8, 8) * 1000).astype('int16')
try:
    if _codecs:
        za = zarr.create_array(store=_test_path, shape=_dummy.shape, chunks=(1,1,4,8,8),
                               dtype='int16', codecs=_codecs)
    else:
        za = zarr.create_array(store=_test_path, shape=_dummy.shape, chunks=(1,1,4,8,8),
                               dtype='int16')
    za[:] = _dummy
    za.attrs.update({'test': True})
    ZARR_WRITE_MODE = 'create_array'
except Exception as e1:
    print("create_array failed:", e1, "| trying zarr.open fallback")
    za = zarr.open(_test_path, mode='w', shape=_dummy.shape, chunks=(1,1,4,8,8), dtype='int16')
    za[:] = _dummy
    za.attrs.update({'test': True})
    ZARR_WRITE_MODE = 'open'

zb = zarr.open(_test_path, mode='r')
assert np.array_equal(zb[:], _dummy), "round-trip mismatch -- values corrupted on write/read"
assert zb.attrs.asdict().get('test') is True, "attrs did not survive the round trip"
print(f"zarr v3 self-test PASSED via '{ZARR_WRITE_MODE}'. Using this method below.")
_sh.rmtree(_test_path)

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

import geotessera
from geotessera.store import GeoTesseraZarr
print("geotessera:", geotessera.__version__)

ROOT        = '/content/drive/MyDrive/Crop_Classification'
DIR_POINTS  = f'{ROOT}/01_Points'
DIR_PATCHES = f'{ROOT}/04_Patches'
DIR_QA      = f'{ROOT}/05_QA'
os.makedirs(DIR_PATCHES, exist_ok=True); os.makedirs(DIR_QA, exist_ok=True)

WORK_DIR = '/content/work_te'
os.makedirs(WORK_DIR, exist_ok=True)

POINTS_GPKG  = f'{DIR_POINTS}/wbcrop_points_extended.gpkg'
POINTS_LAYER = 'wbcrop_points_extended'

gz = GeoTesseraZarr()
print("Years in the store schema:", gz.years)

zarr version OK: 3.3.0
create_array failed: create_array() got an unexpected keyword argument 'codecs' | trying zarr.open fallback
zarr v3 self-test PASSED via 'open'. Using this method below.


/tmp/ipykernel_2028/470924761.py:22: DeprecationWarning: BloscShuffle.bitshuffle is deprecated; pass the string 'bitshuffle' instead.
  _codecs = [BytesCodec(), BloscCodec(cname='zstd', clevel=5, shuffle=BloscShuffle.bitshuffle)]


Mounted at /content/drive
geotessera: 0.10.2
Years in the store schema: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## Step 1 — introspect the library

Kept for reproducibility if a future geotessera version changes the API.

In [3]:
all_attrs = [n for n in dir(gz) if not n.startswith('_')]
print("all public attributes/methods:")
for n in all_attrs:
    print(" ", n)

KEYWORDS = ['tile', 'region', 'window', 'array', 'store', 'zarr', 'read', 'get', 'open',
            'chunk', 'block', 'patch', 'crop', 'bbox', 'bounds']
candidates = [n for n in all_attrs if any(k in n.lower() for k in KEYWORDS)]
print("\nCANDIDATES for a windowed/tile read:", candidates)
print("\nIf one of these looks right (e.g. get_tile, tile_array, open_tile, .store, .zarr),")
print("call help() on it next and use it to replace read_tile_window() below.")

all public attributes/methods:
  build_version
  depths
  iter_region
  model_version
  n_bands
  open_zone
  probe
  read_patch
  read_region
  read_region_quantized
  sample_at
  sample_points
  url
  years

CANDIDATES for a windowed/tile read: ['iter_region', 'open_zone', 'read_patch', 'read_region', 'read_region_quantized']

If one of these looks right (e.g. get_tile, tile_array, open_tile, .store, .zarr),
call help() on it next and use it to replace read_tile_window() below.


## Step 2 — read the real source of `sample_points`

In [4]:
try:
    print(inspect.getsource(GeoTesseraZarr.sample_points))
except (OSError, TypeError) as e:
    print(f"Could not retrieve source ({e}). Try: help(GeoTesseraZarr.sample_points)")
    help(GeoTesseraZarr.sample_points)

    def sample_points(
        self,
        coords: List[Tuple[float, float]],
        year: int,
        *,
        progress: bool = True,
        cross_zone: bool = True,
        search_px: int = SEAM_SEARCH_PX,
        depth: Optional[int] = None,
    ) -> np.ndarray:
        """Sample embeddings at points, routing each to its zone.

        Args:
            coords: List of ``(lon, lat)`` tuples in WGS84.
            cross_zone: See :meth:`sample_at`.
            search_px: See :meth:`sample_at`.
            depth: Matryoshka depth to read, e.g. 16 on a v2 store; the
                first *depth* dimensions arrive for a fraction of the
                bytes.  None reads the full embedding.

        Returns ``(N, B)`` float32, one bulk read per UTM zone, NaN rows
        for points without an embedding.  Unwritten pixels and points
        near a zone seam retry through :meth:`sample_at`.
        ``progress`` is deprecated and ignored: progress is logged
        through the ``geote

## Step 3 — the confirmed reader

In [5]:
def read_tile_window(lon, lat, year, size_px):
    """gz.read_patch, confirmed via introspection above. Returns (array[B,H,W], transform, crs).
    transform is a rasterio.transform.Affine; crs is a string. Both can legitimately differ
    from one point to the next near a UTM-zone seam -- store them per sample, not per tile."""
    patch, transform, crs = gz.read_patch(lon, lat, year, size_px)   # (H, W, B) float32
    return np.transpose(patch, (2, 0, 1)), transform, crs            # -> (B, H, W)

## Verify, and measure the real value range

One point, one year. This sets `SCALE_FACTOR` from what actually comes back rather than an
assumption. If this cell does not produce a plausible array, **stop** — do not proceed.

In [6]:
pts = gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER).to_crs('EPSG:4326')
assert pts['id'].is_unique, "duplicate ids in the points file"

TEST_YEAR = 2024
PROBE_PATCH = 128
lon0, lat0 = float(pts.geometry.x.iloc[0]), float(pts.geometry.y.iloc[0])

arr, transform, crs = read_tile_window(lon0, lat0, TEST_YEAR, PROBE_PATCH)
print("shape:", arr.shape, "(band, y, x) | crs:", crs)
print("transform:", list(transform)[:6])

finite = np.isfinite(arr)
print(f"\nfinite fraction: {finite.mean():.1%}")
vmax_abs = float(np.nanmax(np.abs(arr))) if finite.any() else None
print("value range:", float(np.nanmin(arr)), "to", float(np.nanmax(arr)),
      "| max abs:", vmax_abs)

assert vmax_abs and np.isfinite(vmax_abs), "no finite pixels in the probe patch -- try a different point"
MEASURED_SCALE_FACTOR = vmax_abs / 32000.0    # headroom below int16 max so nothing clips
print(f"\nMEASURED_SCALE_FACTOR = {MEASURED_SCALE_FACTOR:.6g}  (int16, no assumed range)")
print("Carried into SCALE_FACTOR in the config cell below.")

shape: (128, 128, 128) (band, y, x) | crs: EPSG:32645
transform: [10.0, 0.0, 776370.0, 0.0, -10.0, 2936690.0]

finite fraction: 100.0%
value range: -12.884605407714844 to 11.531327247619629 | max abs: 12.884605407714844

MEASURED_SCALE_FACTOR = 0.000402644  (int16, no assumed range)
Carried into SCALE_FACTOR in the config cell below.


## Configuration

`TILE_DEG` is output-grouping only — `read_patch` doesn't need it.

In [7]:
TEST_MODE      = True
TEST_N_POINTS  = 15
TEST_TILE_RANK = 0

CANDIDATE_YEARS = [2022, 2023, 2024, 2025]   # measured below; only covered years are kept
PATCH    = 128
TILE_DEG = 0.1     # output-file grouping only, not a read constraint

NODATA       = -32768
SCALE_FACTOR = MEASURED_SCALE_FACTOR     # from the probe cell above -- not assumed
MAX_RETRIES  = 3
FORCE        = False

RUN_ID = f"tessera_{PATCH}px_{'_'.join(map(str, CANDIDATE_YEARS))}"
if TEST_MODE:
    RUN_ID += "_TEST"
STORE_DIR = os.path.join(DIR_PATCHES, RUN_ID)
os.makedirs(STORE_DIR, exist_ok=True)
print("STORE:", STORE_DIR)

STORE: /content/drive/MyDrive/Crop_Classification/04_Patches/tessera_128px_2022_2023_2024_2025_TEST


## Measure real year coverage

In [8]:
pts['tx'] = np.floor(pts.geometry.x / TILE_DEG).astype(int)
pts['ty'] = np.floor(pts.geometry.y / TILE_DEG).astype(int)
rep = pts.groupby(['tx','ty']).first().reset_index()[['tx','ty','geometry']]
rep_coords = np.array([[g.x, g.y] for g in rep.geometry])

year_cov = {}
for y in CANDIDATE_YEARS:
    if y not in gz.years:
        print(f"  {y}  not in schema"); continue
    try:
        X = gz.sample_points(rep_coords, year=y)
        ok = np.isfinite(np.asarray(X)).any(axis=1)
        year_cov[y] = float(ok.mean())
        print(f"  {y}  tile coverage: {ok.sum():>4}/{len(ok)} ({ok.mean():>5.1%})")
    except Exception as e:
        print(f"  {y}  FAIL  {type(e).__name__}: {str(e)[:60]}")

MIN_TILE_COVERAGE = 0.90
YEARS = [y for y, c in year_cov.items() if c >= MIN_TILE_COVERAGE]
print("\nYears kept for patch extraction:", YEARS)
assert YEARS, "no year clears the coverage threshold -- cannot proceed"

  2022  tile coverage:   36/511 ( 7.0%)
  2023  tile coverage:   43/511 ( 8.4%)
  2024  tile coverage:  511/511 (100.0%)
  2025  tile coverage:    0/511 ( 0.0%)

Years kept for patch extraction: [2024]


## Points and tiling

In [9]:
tiles_all = pts.groupby(['tx','ty']).size().sort_values(ascending=False)

if TEST_MODE:
    tkey = tiles_all.index[TEST_TILE_RANK]
    pts_run = pts[(pts['tx']==tkey[0]) & (pts['ty']==tkey[1])].head(TEST_N_POINTS).copy()
    print(f"TEST: tile {tkey}, {len(pts_run)} points")
else:
    pts_run = pts.copy()

tiles = pts_run.groupby(['tx','ty']).size().sort_values(ascending=False)
N_INPUT = len(pts_run)
print(f"Points: {N_INPUT:,} | tiles: {len(tiles)}")
# No utm_epsg() helper needed: read_patch() returns each sample's own crs directly.

TEST: tile (np.int64(882), np.int64(265)), 15 points
Points: 15 | tiles: 1


## Extract one tile

`crs`/`transform` read once (year[0]) and reused — zone routing depends on location, not year. Per-point failures write an all-nodata patch rather than aborting the tile.

In [10]:
def process_tile(tx, ty, tp):
    zip_path = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.zarr.zip")
    meta_p   = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.meta.json")
    if os.path.exists(meta_p) and os.path.exists(zip_path) and not FORCE:
        return 'skipped'

    n_p, n_t = len(tp), len(YEARS)

    first, _, _ = read_tile_window(float(tp.geometry.x.iloc[0]),
                                   float(tp.geometry.y.iloc[0]), YEARS[0], PATCH)
    n_b = first.shape[0]

    local = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")
    if os.path.exists(local):
        shutil.rmtree(local)
    shape  = (n_p, n_t, n_b, PATCH, PATCH)
    chunks = (1, 1, n_b, PATCH, PATCH)
    if ZARR_WRITE_MODE == 'create_array' and _codecs:
        z = zarr.create_array(store=local, shape=shape, chunks=chunks,
                              dtype='int16', codecs=_codecs)
    elif ZARR_WRITE_MODE == 'create_array':
        z = zarr.create_array(store=local, shape=shape, chunks=chunks, dtype='int16')
    else:
        z = zarr.open(local, mode='w', shape=shape, chunks=chunks, dtype='int16')

    ids = [int(v) for v in tp['id'].values]
    samples = []
    for pi in range(n_p):
        lon, lat = float(tp.geometry.x.iloc[pi]), float(tp.geometry.y.iloc[pi])
        transform, crs = None, None
        for ti, year in enumerate(YEARS):
            try:
                arr, tr, cr = read_tile_window(lon, lat, year, PATCH)
                if transform is None:      # geometry is year-independent; keep the first
                    transform, crs = tr, cr
            except Exception:
                arr = np.full((n_b, PATCH, PATCH), np.nan, dtype='float32')
            bad = ~np.isfinite(arr)
            enc = np.clip(np.where(bad, 0, arr / SCALE_FACTOR), -32767, 32767)
            enc = np.where(bad, NODATA, enc).astype('int16')
            z[pi, ti] = enc

        samples.append({
            'id': ids[pi], 'array_index': pi,
            # per-sample: a point near a UTM-zone seam gets its own merged projection,
            # so there is no single tile-wide crs/transform the way MPC tiles have one.
            'crs': crs, 'transform': list(transform)[:6] if transform is not None else None,
            'center_lonlat': [lon, lat],
            'crop': str(tp['crop'].values[pi]), 'district': str(tp['district'].values[pi]),
        })

    z.attrs.update({
        'run_id': RUN_ID, 'source': 'tessera',
        'dims': ['point','year','band','y','x'], 'bands': [f'T{i:03d}' for i in range(n_b)],
        'dtype': 'int16', 'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
        'crs': 'PER-SAMPLE -- see samples[i]["crs"], not one CRS per tile',
        'patch': PATCH, 'years': YEARS, 'point_ids': ids,
        'transform_convention': 'affine.Affine a,b,c,d,e,f: x=a*col+b*row+c, y=d*col+e*row+f',
        'decode': 'value * scale_factor where value != nodata',
    })
    tmp_zip = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")
    shutil.make_archive(tmp_zip, 'zip', local)
    shutil.move(tmp_zip + '.zip', zip_path)
    shutil.rmtree(local)

    tile_meta = {
        'run_id': RUN_ID, 'source': 'tessera', 'tile': [int(tx), int(ty)],
        'store': os.path.basename(zip_path), 'shape': [n_p, n_t, n_b, PATCH, PATCH],
        'dims': ['point','year','band','y','x'], 'bands': [f'T{i:03d}' for i in range(n_b)],
        'dtype': 'int16', 'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
        'crs': 'PER-SAMPLE -- see each entry in samples', 'patch': PATCH,
        'n_points': n_p, 'years': YEARS, 'point_ids': ids, 'samples': samples,
    }
    with open(meta_p, 'w') as f:
        json.dump(tile_meta, f, indent=1)

    pd.DataFrame({'id': ids, 'array_index': np.arange(n_p),
                  'store': os.path.basename(zip_path),
                  'crop': tp['crop'].values, 'district': tp['district'].values,
                  'crs': [s['crs'] for s in samples], 'tile_tx': tx, 'tile_ty': ty}
                 ).to_parquet(os.path.join(STORE_DIR, f"tile_{tx}_{ty}.points.parquet"),
                               index=False)
    return 'done'


def process_retry(tx, ty, tp):
    for a in range(1, MAX_RETRIES + 1):
        try:
            return process_tile(tx, ty, tp)
        except Exception as e:
            if a == MAX_RETRIES:
                print(f"  tile {tx}_{ty} FAILED: {type(e).__name__}: {str(e)[:90]}")
                return 'error'
            time.sleep(2 ** a)
    return 'error'

## Run

In [11]:
if TEST_MODE:
    (tx0, ty0) = tiles.index[0]
    t0 = time.time()
    r  = process_retry(tx0, ty0, pts_run[(pts_run['tx']==tx0)&(pts_run['ty']==ty0)])
    dt = time.time() - t0
    zip0 = os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.zarr.zip")
    print("result:", r, f"| {dt:.0f}s")
    if r == 'done':
        mb = os.path.getsize(zip0) / 1e6
        m  = json.load(open(os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.meta.json")))
        print(f"{m['n_points']} points x {len(m['years'])} years -> {mb:,.1f} MB")
        proj_gb = mb / m['n_points'] * len(pts) / 1e3
        proj_hr = dt / m['n_points'] * len(pts) / 3600
        print(f"\nPROJECTED full run ({len(pts):,} points): "
              f"~{proj_gb:,.1f} GB | ~{proj_hr:,.1f} hours")
        print("If this is slow, it is almost certainly because Step 3 is calling")
        print("sample_points once per point rather than reading a real tile window --")
        print("go back to Steps 1-2 and look for a batched/region accessor.")
else:
    failed = []
    stats = {'done':0,'skipped':0,'error':0}
    t0 = time.time()
    for i, ((tx, ty), cnt) in enumerate(tiles.items(), 1):
        r = process_retry(tx, ty, pts_run[(pts_run['tx']==tx)&(pts_run['ty']==ty)])
        stats[r] += 1
        if r == 'error':
            failed.append({'tx': int(tx), 'ty': int(ty)})
        if i % 20 == 0 or i == len(tiles):
            el = time.time() - t0
            print(f"[{i}/{len(tiles)}] {stats} | {el/60:.1f} min")

result: done | 19s
15 points x 1 years -> 60.0 MB

PROJECTED full run (74,351 points): ~297.6 GB | ~26.1 hours
If this is slow, it is almost certainly because Step 3 is calling
sample_points once per point rather than reading a real tile window --
go back to Steps 1-2 and look for a batched/region accessor.


## Audit

In [12]:
stores = sorted(glob.glob(os.path.join(STORE_DIR, "tile_*.zarr.zip")))
assert stores, "no store written"
s0 = stores[0]
from zarr.storage import ZipStore
zs = ZipStore(s0, mode='r')     # v3: ZipStore moved under zarr.storage
z  = zarr.open(zs, mode='r')
A  = z.attrs.asdict()
n_p, n_t, n_b = z.shape[0], z.shape[1], z.shape[2]
print("array shape:", z.shape, "| years:", A['years'])
assert n_t == len(YEARS), f"expected {len(YEARS)} year-steps, got {n_t}"

valid = np.array([[float((z[pi, ti] != A['nodata']).mean()) for ti in range(n_t)]
                  for pi in range(n_p)])
print(f"valid-pixel fraction: min {valid.min():.2f} | median {np.median(valid):.2f}")

M = json.load(open(s0.replace(".zarr.zip", ".meta.json")))
ok = (M['years'] == YEARS and len(M['samples']) == n_p)
print("JSON agrees with the array:", ok)
assert ok

# CRS is per-sample here (unlike the other patch notebooks). Most points should share the
# tile's dominant UTM zone; any that differ sat near a zone seam and got merged onto their
# own projection by read_patch -- expected, not an error, but worth seeing how many.
crs_list = [s['crs'] for s in M['samples']]
counts = pd.Series(crs_list).value_counts()
print("\nCRS distribution across this tile's samples:")
print(counts.to_string())
if len(counts) > 1:
    print(f"\n{counts.iloc[1:].sum()} of {n_p} samples used a non-dominant/merged CRS "
          "(zone-seam points) -- expected for points near a UTM boundary.")
missing_geo = sum(1 for s in M['samples'] if s['crs'] is None)
print(f"samples with no transform/crs at all (every year failed): {missing_geo}")

array shape: (15, 1, 128, 128, 128) | years: [2024]
valid-pixel fraction: min 1.00 | median 1.00
JSON agrees with the array: True

CRS distribution across this tile's samples:
EPSG:32645    15
samples with no transform/crs at all (every year failed): 0


In [13]:
qa = {
    'generated': pd.Timestamp.now().isoformat(),
    'mode': 'TEST' if TEST_MODE else 'FULL',
    'run_id': RUN_ID, 'source': 'tessera', 'years': YEARS,
    'year_coverage': year_cov, 'patch': PATCH, 'dtype': 'int16',
    'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
    'points': int(n_p), 'valid_fraction_median': float(np.median(valid)),
    'window_read_method': 'SEE Step 1/2 output for the accessor actually used',
}
with open(os.path.join(DIR_QA, f'07_patches_{RUN_ID}_qa.json'), 'w') as f:
    json.dump(qa, f, indent=2, default=str)
print("QA saved.")

QA saved.


---
### Before `TEST_MODE = False`
- Check `MEASURED_SCALE_FACTOR` looks sane.
- Audit: years present, valid-pixel fractions reasonable, CRS distribution (a few non-dominant = zone-seam points, expected).
- Check the size/time projection — `read_patch` caches zones, so it should beat a naive per-point cost model.
- If geotessera changes, re-run Step 1's introspection first.